# Input metadata checks

Inspect vector and raster metadata before running overlay or zonal summaries. Update the configuration block for the local input data used in a new application.


In [ ]:
from pathlib import Path

# Run from the repository root or from the notebooks folder.
HERE = Path.cwd()
ROOT = HERE.parent if HERE.name == "notebooks" else HERE

DATA = ROOT / "data_raw"
OUTPUTS = ROOT / "outputs"
TABLES = ROOT / "tables"
FIGURES = ROOT / "figures"

for folder in (OUTPUTS, TABLES, FIGURES):
    folder.mkdir(parents=True, exist_ok=True)

import json
from datetime import datetime, timezone

import geopandas as gpd
import pandas as pd
import rasterio

# ---------------------------------------------------------------------
# Configuration: update these paths for a new study area.
# ---------------------------------------------------------------------
BOUNDARY_FILE = DATA / "boundary.gpkg"
POPULATION_RASTER = DATA / "population.tif"
CLASS_RASTER = DATA / "settlement_or_landcover.tif"

INPUTS = {
    "boundary": BOUNDARY_FILE,
    "population_raster": POPULATION_RASTER,
    "categorical_raster": CLASS_RASTER,
}

def vector_metadata(path: Path) -> dict:
    gdf = gpd.read_file(path)
    bounds = [float(x) for x in gdf.total_bounds]
    return {
        "path": str(path.relative_to(ROOT)),
        "type": "vector",
        "features": int(len(gdf)),
        "crs": str(gdf.crs),
        "bounds": bounds,
        "geometry_types": sorted(map(str, gdf.geometry.geom_type.dropna().unique())),
    }

def raster_metadata(path: Path) -> dict:
    with rasterio.open(path) as src:
        return {
            "path": str(path.relative_to(ROOT)),
            "type": "raster",
            "width": int(src.width),
            "height": int(src.height),
            "count": int(src.count),
            "crs": str(src.crs),
            "bounds": [float(src.bounds.left), float(src.bounds.bottom), float(src.bounds.right), float(src.bounds.top)],
            "resolution": [float(abs(src.res[0])), float(abs(src.res[1]))],
            "nodata": None if src.nodata is None else float(src.nodata),
            "dtype": src.dtypes[0],
        }

records = []
for name, path in INPUTS.items():
    if not path.exists():
        records.append({"name": name, "path": str(path.relative_to(ROOT)), "status": "missing"})
        continue

    if path.suffix.lower() in {".tif", ".tiff"}:
        meta = raster_metadata(path)
    else:
        meta = vector_metadata(path)
    meta["name"] = name
    meta["status"] = "found"
    records.append(meta)

summary = {
    "run_time_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "records": records,
}

with open(OUTPUTS / "input_metadata_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

pd.DataFrame(records).to_csv(TABLES / "input_metadata_summary.csv", index=False)
pd.DataFrame(records)
